In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


#Part B — Predictive modeling, continuing from the same cleaned data

#Task-7

Split the data into train/test sets first, using a stratified split (justify why stratification matters given the class balance you observed in Task 1). Use survived as the classification target.

In [25]:
df_titanic=pd.read_csv("/content/titanic_cleaned.csv")

# Separate features and target
X = df_titanic.drop(columns=['survived'])
y = df_titanic['survived']

#Split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

print(f"Train set shape: {X_train.shape}, Test set shape: {X_test.shape}")
print(f"Train survival rate: {y_train.mean():.4f}, Test survival rate: {y_test.mean():.4f}")


Train set shape: (711, 7), Test set shape: (178, 7)
Train survival rate: 0.3826, Test survival rate: 0.3820


#Task-8

Preprocessing (fit on training data only): handle missing values in the columns you use (you do not need to match your Task 2 strategy exactly, but state your choice), encode categorical columns (sex, embarked) with label or one-hot encoding, and scale numeric features with StandardScaler. Every preprocessing step (imputer, encoder, scaler) must be fit only on the training split, then applied in transform-only mode to the test split — never fit or refit any preprocessing step on the test data or on the full pre-split dataset, since that leaks test-set information into training. It is strongly recommended you implement this with a scikit-learn Pipeline/ColumnTransformer (a ColumnTransformer for per-column imputing/encoding/scaling, wrapped in a Pipeline with the final estimator) so the fit-on-train / transform-on-test separation is enforced structurally rather than left for you to remember by hand.

In [26]:
from numpy import median
numeric_features = ['age', 'fare', 'pclass', 'sibsp', 'parch']
categorical_features = ['sex', 'embarked']

numerical_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(drop='first'))
])

preprocessor = ColumnTransformer([
    ('num',numerical_transformer,numeric_features),
    ('cat',categorical_transformer,categorical_features)
])


#Task-9

Train three classifiers on the same train/test split: Logistic Regression, Decision Tree, and Random Forest. For the Decision Tree, additionally render it with plot_tree, labeling feature names and class names.

In [27]:
classifiers={
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4,random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100,random_state=42,oob_score=True)
}


#Task-10

Evaluate all three models with: a confusion matrix, accuracy, precision, recall, F1 score, and an ROC curve with AUC. Present these side by side in a single comparison table.

In [28]:
results=[]
plt.figure(figsize=(8,6))

for name,clf in classifiers.items():
  pipe = Pipeline([
      ('preprocessor',preprocessor),
      ('classifier',clf)
  ])

  # Fit strictly on train split
  pipe.fit(X_train,y_train)
  y_pred = pipe.predict(X_test)
  y_proba = pipe.predict_proba(X_test)[:, 1]
  cm = confusion_matrix(y_test,y_pred)
  acc = accuracy_score(y_test,y_pred)
  prec = precision_score(y_test,y_pred)
  rec = recall_score(y_test,y_pred)
  f1 = f1_score(y_test,y_pred)
  auc = roc_auc_score(y_test,y_proba)

  results.append({
        'Model': name, 'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1 Score': f1, 'ROC AUC': auc, 'Confusion Matrix': cm.tolist()
    })

  # ROC Plotting
  fpr, tpr, _ = roc_curve(y_test, y_proba)
  plt.plot(fpr, tpr, label = f"{name} (AUC : {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label="Random Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Classification Models")
plt.legend()
plt.tight_layout()
plt.savefig("/content/06_roc_curves.png", dpi=300)
plt.close()

# Visualize Decision Tree
dt_pipe=Pipeline([
    ('preprocessor', preprocessor),
    ('clf', DecisionTreeClassifier(max_depth=3,random_state=42))
])
dt_pipe.fit(X_train, y_train)
ohe_feature_names = dt_pipe.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(ohe_feature_names)

plt.figure(figsize=(16, 8))
plot_tree(dt_pipe.named_steps['clf'], feature_names=all_feature_names, class_names=['Died', 'Survived'], filled=True, fontsize=9)
plt.title("Decision Tree Visualization (Max Depth = 3)")
plt.tight_layout()
plt.savefig("/content/05_decision_tree.png", dpi=300)
plt.close()


#Task-11

Imbalance handling comparison: report survived/not-survived class balance, then retrain (any one of the three models is enough for this sub-task) three ways — (a) baseline/no handling, (b) class_weight='balanced', (c) SMOTE oversampling applied only to the training fold (to avoid leakage) — and compare precision/recall/F1 across the three variants, with a short written conclusion on which imbalance strategy worked best and why.

In [30]:
# (a) Baseline
rf_baseline = Pipeline([('preprocessor', preprocessor), ('clf', RandomForestClassifier(random_state=42))])
rf_baseline.fit(X_train, y_train)
y_pred_base = rf_baseline.predict(X_test)

# (b) Class Weight Balanced
rf_balanced = Pipeline([('preprocessor', preprocessor), ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))])
rf_balanced.fit(X_train, y_train)
y_pred_bal = rf_balanced.predict(X_test)

# (c) SMOTE (Applied ONLY inside training fold)
rf_smote = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(random_state=42))
])
rf_smote.fit(X_train, y_train)
y_pred_smote = rf_smote.predict(X_test)

imbalance_results = pd.DataFrame([
    {'Strategy': 'Baseline (None)', 'Precision': precision_score(y_test, y_pred_base), 'Recall': recall_score(y_test, y_pred_base), 'F1 Score': f1_score(y_test, y_pred_base)},
    {'Strategy': "class_weight='balanced'", 'Precision': precision_score(y_test, y_pred_bal), 'Recall': recall_score(y_test, y_pred_bal), 'F1 Score': f1_score(y_test, y_pred_bal)},
    {'Strategy': 'SMOTE (Train fold only)', 'Precision': precision_score(y_test, y_pred_smote), 'Recall': recall_score(y_test, y_pred_smote), 'F1 Score': f1_score(y_test, y_pred_smote)}
])
print(imbalance_results.to_string(index=False))


               Strategy  Precision   Recall  F1 Score
        Baseline (None)   0.727273 0.705882  0.716418
class_weight='balanced'   0.750000 0.705882  0.727273
SMOTE (Train fold only)   0.712121 0.691176  0.701493


#Task-12

Hyperparameter tuning: run GridSearchCV over the Random Forest's n_estimators, max_depth, and max_features, report the best parameter combination and the corresponding out-of-bag (OOB) score. Because oob_score_ is only populated when oob_score=True is passed at construction time, you must construct the estimator as RandomForestClassifier(oob_score=True, ...) (together with your other chosen/tuned parameters) — otherwise the OOB score will not be available to report.

In [32]:
rf_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [4, 6, 8, None],
    'classifier__max_features': ['sqrt', 'log2']
}

rf_tune_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(oob_score=True, random_state=42))
])

grid_search = GridSearchCV(rf_tune_pipe, param_grid=rf_param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_rf = best_model.named_steps['classifier']

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Out-of-Bag (OOB) Score: {best_rf.oob_score_:.4f}")


Best Parameters: {'classifier__max_depth': 8, 'classifier__max_features': 'log2', 'classifier__n_estimators': 100}
Out-of-Bag (OOB) Score: 0.8158


#Task-13

Regression side-task: using the same dataset, predict fare from the other available features with a multivariate linear regression. Report MAE, RMSE, R², and Adjusted R², and produce a residual plot, stating in writing whether it shows heteroscedasticity (a non-random spread of residuals).

In [34]:
reg_features = ['age', 'pclass', 'sibsp', 'parch', 'survived', 'sex', 'embarked']
X_reg = df_titanic[reg_features]
y_reg = df_titanic['fare']

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)

reg_num_cols = ['age', 'pclass', 'sibsp', 'parch', 'survived']
reg_cat_cols = ['sex', 'embarked']

reg_preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), reg_num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(drop='first'))]), reg_cat_cols)
])

reg_pipeline = Pipeline([
    ('preprocessor', reg_preprocessor),
    ('regressor', LinearRegression())
])

reg_pipeline.fit(X_reg_train, y_reg_train)
y_reg_pred = reg_pipeline.predict(X_reg_test)

mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2 = r2_score(y_reg_test, y_reg_pred)

n = len(y_reg_test)
p = X_reg_train.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"Regression Metrics -> MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f} | Adj R²: {adj_r2:.4f}")

# Residual Plot
residuals = y_reg_test - y_reg_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_reg_pred, residuals, alpha=0.6, color='purple')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Fare ($)")
plt.ylabel("Residuals ($)")
plt.title("Linear Regression Residual Plot (Fare Prediction)")
plt.tight_layout()
plt.savefig("/content/07_regression_residuals.png", dpi=300)
plt.close()


Regression Metrics -> MAE: 21.1238 | RMSE: 41.6843 | R²: 0.3487 | Adj R²: 0.3219


#Task-14 AND Task-15

14.
Write a model comparison table that presents the three classifiers' metrics (accuracy, precision, recall, F1, AUC) side by side, and the regression model's metrics (MAE, RMSE, R², Adjusted R²) side by side as their own separate columns. Classification metrics and regression metrics are on different scales and are not directly comparable numbers — the table must present them as two distinct metric groups (one per model type), not implied to be on a single shared scale. Add a 3–5 sentence final written recommendation of which classifier you would deploy and why, referencing specific metric values.

15.
Save your best-performing complete pipeline — the fitted preprocessing steps (imputer/encoder/scaler, or your ColumnTransformer) together with the final estimator, as a single combined object (e.g. a scikit-learn Pipeline) — to disk using joblib.dump(full_pipeline, ...). Do not save the bare estimator alone: the saved artifact must be usable end-to-end on raw, unpreprocessed new data. Include a short script/cell that reloads it with joblib.load and confirms it still predicts correctly on raw input.

In [36]:
# Save Best Complete Pipeline (Preprocessing + Estimator)
artifact_path = "/content/titanic_pipeline.joblib"
joblib.dump(best_model, artifact_path)
print(f"\nSuccessfully persisted complete end-to-end pipeline to '{artifact_path}'.")

# Verification Reloading Cell
loaded_pipeline = joblib.load(artifact_path)
raw_sample = pd.DataFrame([{
    'pclass': 1, 'sex': 'female', 'age': 28.0, 'sibsp': 0, 'parch': 0,
    'fare': 150.0, 'embarked': 'C', 'who': 'woman', 'adult_male': False,
    'alone': True, 'class': 'First', 'embark_town': 'Cherbourg', 'alive': 'yes'
}])

sample_pred = loaded_pipeline.predict(raw_sample)
sample_proba = loaded_pipeline.predict_proba(raw_sample)[0, 1]

print("\n--- Pipeline Reload & Inference Check ---")
print(f"Raw Sample Prediction: {'Survived' if sample_pred[0] == 1 else 'Died'} (Probability: {sample_proba:.4f})")



Successfully persisted complete end-to-end pipeline to '/content/titanic_pipeline.joblib'.

--- Pipeline Reload & Inference Check ---
Raw Sample Prediction: Survived (Probability: 0.9967)
